# Copilot Org Data — Direct Ingester (Fabric)

End-to-end Lakehouse loader for **org / HRIS data** (manager, department, location) that calls Microsoft Graph directly from inside Fabric. Replaces the prior flow:

```
OLD:  PowerShell → CSV → Files/org_raw/ → Copilot_Org_Data_Loader.ipynb → Delta
NEW:  This notebook (Graph → Delta)
```

**Source endpoint**: `/v1.0/users?$select=...&$expand=manager($select=userPrincipalName)&$top=999` — pages through every Entra user with org-relevant fields and resolves manager UPN inline.

**Output**: Lakehouse Delta table `dbo.copilot_org_data` (same name + schema as the existing CSV loader, so the PBIT works without changes).

**Permissions**: app registration with `User.Read.All` (Application permission, admin-consented).


## 1. Configuration

In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'

OUTPUT_TABLE  = 'dbo.copilot_org_data'
WRITE_MODE    = 'overwrite'


## 2. Authenticate to Microsoft Graph

In [ ]:
import requests

def get_graph_token(tenant_id, client_id, client_secret):
    url  = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    body = {
        'client_id':     client_id,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

token   = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
headers = {'Authorization': f'Bearer {token}'}
print('✓ Graph token acquired.')


## 3. Page through `/users` with manager expand

Graph caps `$top` at 999. Follow `@odata.nextLink` until exhausted.


In [ ]:
select_fields  = 'id,userPrincipalName,displayName,department,jobTitle,companyName,officeLocation,city,country,accountEnabled'
expand_manager = 'manager($select=userPrincipalName)'
next_url = f'https://graph.microsoft.com/v1.0/users?$select={select_fields}&$expand={expand_manager}&$top=999'

users = []
while next_url:
    r = requests.get(next_url, headers=headers, timeout=60)
    r.raise_for_status()
    data = r.json()
    users.extend(data.get('value', []))
    next_url = data.get('@odata.nextLink')
    print(f'  Fetched {len(users):,} users so far...')

print(f'✓ Total users: {len(users):,}')


## 4. Flatten manager UPN + canonicalise columns

`manager` is a nested object — flatten to `managerUPN`. Then rename to match the existing loader's canonical schema (`PersonId`, `Organization`, `JobTitle`).


In [ ]:
rows = []
for u in users:
    manager_upn = ''
    if u.get('manager') and isinstance(u['manager'], dict):
        manager_upn = u['manager'].get('userPrincipalName', '') or ''
    rows.append({
        'id':               u.get('id'),                  # AAD objectId — join key for Dataverse user_id_hash
        'PersonId':         u.get('userPrincipalName'),    # canonical: PersonId (was userPrincipalName)
        'displayName':      (u.get('displayName') or u.get('userPrincipalName')),  # coalesce: Graph returns null displayName for rooms/shared/service accounts
        'Organization':     u.get('department'),           # canonical: Organization (was department)
        'JobTitle':         u.get('jobTitle'),             # canonical: JobTitle
        'companyName':      u.get('companyName'),
        'officeLocation':   u.get('officeLocation'),
        'city':             u.get('city'),
        'country':          u.get('country'),
        'accountEnabled':   str(u.get('accountEnabled', '')),
        'managerUPN':       manager_upn,
    })

print(f'Built {len(rows):,} canonicalised rows.')


## 5. Build Spark DataFrame + add normalised key + sanitise columns

Uses an **explicit schema** because Graph's `/users` response can have all-`None` values for some columns on small tenants (e.g. nobody has a `country` set), and Spark's automatic schema inference fails on those with `[CANNOT_DETERMINE_TYPE]`.


In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Explicit string schema — avoids inference failure on small/empty-column tenants.
schema = StructType([
    StructField('id',              StringType(), True),
    StructField('PersonId',        StringType(), True),
    StructField('displayName',     StringType(), True),
    StructField('Organization',    StringType(), True),
    StructField('JobTitle',        StringType(), True),
    StructField('companyName',     StringType(), True),
    StructField('officeLocation',  StringType(), True),
    StructField('city',            StringType(), True),
    StructField('country',         StringType(), True),
    StructField('accountEnabled',  StringType(), True),
    StructField('managerUPN',      StringType(), True),
])

df = spark.createDataFrame(rows, schema=schema)

# Add normalised join key (matches existing loader)
df = df.withColumn(
    'PersonId_Normalized',
    F.when(F.col('PersonId').isNull(), None)
     .otherwise(F.lower(F.trim(F.col('PersonId').cast('string'))))
)

# Add TotalEmployees as a string column (matches existing loader)
total = df.count()
df = df.withColumn('TotalEmployees', F.lit(str(total)))

# Sanitise column names for Delta
_INVALID = re.compile(r'[ ,;{}()\n\t=]')
df = df.toDF(*[_INVALID.sub('_', c) for c in df.columns])

print(f'Final rows: {df.count():,}')
print('Columns:', df.columns)


## 6. Write to Lakehouse Delta table

In [ ]:
(df.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,}')


## 7. Verify

In [ ]:
tbl = spark.table(OUTPUT_TABLE)
tbl.select('PersonId', 'PersonId_Normalized', 'Organization', 'JobTitle', 'managerUPN').show(10, truncate=False)
tbl.groupBy('Organization').count().orderBy(F.desc('count')).show(20, truncate=False)


---
**Connect the PBIT**: this table is consumed by the `Chat + Agent Org Data` query in both AI-in-One and AI Business Value dashboards. Once this notebook has run, leave the `Org Data File` parameter blank when opening the PBIT — refresh sources from `dbo.copilot_org_data` directly via the Fabric SQL endpoint.
